# C01 · Sistema de Mantenimiento Preventivo de Bombas de Riego
**Introducción a la Inteligencia Artificial (3.4.257) · UADE 2026**
* **Estudiante:** Ariel Martinez
* **Pista:** Neurona artificial (Perceptrón simple)
* **IA declarada:** Gemini
* **Repositorio:** https://github.com/ArielMartinez3/hackaton-iia-uade-2026

### 1. Carga de datos y selección de variables legítimas
Cargamos el dataset oficial `C01.csv` desde el repositorio.
**Decisión metodológica:** Excluimos `repuestos_pedidos_30d` porque representa un hecho posterior a la inspección mensual (Target Leakage). El modelo solo utilizará `horas_uso`, `temperatura` y `vibracion`.


In [ ]:
import pandas as pd
import numpy as np

# Carga directa desde el repositorio de GitHub
url_github = "https://raw.githubusercontent.com/ArielMartinez3/hackaton-iia-uade-2026/main/datos/C01.csv"

try:
    df = pd.read_csv(url_github)
    print("Datos cargados desde GitHub")
except Exception:
    print("Cargando dataset local C01")
    data_c01 = {
        'caso': list(range(1, 21)),
        'conjunto': ['entrenamiento']*14 + ['prueba']*6,
        'horas_uso': [4.1, 9.5, 11.8, 3.2, 7.4, 12.6, 8.9, 5.6, 10.2, 6.3, 2.7, 6.8, 13.4, 4.8, 9.1, 10.7, 3.9, 7.7, 12.1, 5.1],
        'temperatura': [58, 66, 84, 52, 71, 61, 88, 63, 74, 91, 55, 79, 77, 69, 70, 86, 60, 81, 76, 57],
        'vibracion': [2.3, 3.9, 6.8, 1.6, 5.2, 2.9, 7.4, 3.1, 4.4, 8.1, 2.0, 5.9, 5.6, 4.0, 4.6, 7.0, 2.5, 6.4, 6.3, 2.2],
        'repuestos_pedidos_30d': [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
        'resultado_real': [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
    }
    df = pd.DataFrame(data_c01)

# Estandarizamos los nombres de las 7 columnas del CSV para evitar KeyErrors
df.columns = ['caso', 'conjunto', 'horas_uso', 'temperatura', 'vibracion', 'repuestos_30d', 'resultado_real']

# Aseguramos conversión numérica correcta por si los decimales vinieron con coma ('4,1')
for col in ['horas_uso', 'temperatura', 'vibracion']:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)
    else:
        df[col] = df[col].astype(float)

# Normalizamos el texto de la partición (minúsculas y sin espacios)
df['conjunto'] = df['conjunto'].astype(str).str.lower().str.strip()

features = ['horas_uso', 'temperatura', 'vibracion']
entrenamiento = df[df.conjunto == 'entrenamiento'].copy()
prueba = df[df.conjunto == 'prueba'].copy()

print(f"Casos entrenamiento: {len(entrenamiento)} | Casos prueba: {len(prueba)}")
print("Variables de entrada:", features)
df[['caso', 'conjunto'] + features + ['resultado_real']].head(5)


Datos cargados desde GitHub
Casos entrenamiento: 14 | Casos prueba: 6
Variables de entrada: ['horas_uso', 'temperatura', 'vibracion']


,caso,conjunto,horas_uso,temperatura,vibracion,resultado_real
0,1,entrenamiento,4.1,58.0,2.3,0
1,2,entrenamiento,9.5,66.0,3.9,0
2,3,entrenamiento,11.8,84.0,6.8,1
3,4,entrenamiento,3.2,52.0,1.6,0
4,5,entrenamiento,7.4,71.0,5.2,0


### 2. Definición del Perceptrón Simple
Implementamos la combinación lineal con pesos $w = [2.0, 1.0, 10.0]$ y sesgo $b = -155.0$:
$$n = 2 \cdot 	ext{horas} + 1 \cdot 	ext{temperatura} + 10 \cdot 	ext{vibración} - 155.0$$
Regla de decisión: $\hat{y} = 1$ (SÍ) si $n \ge 0.0$; caso contrario $\hat{y} = 0$ (NO).


In [ ]:
W = np.array([2.0, 1.0, 10.0])  # Pesos: [horas, temp, vibracion]
B = -155.0                       # Sesgo base calibrado para priorizar Sensibilidad

def calcular_n(horas, temp, vib, w=W, b=B):
    return w[0] * horas + w[1] * temp + w[2] * vib + b

def clasificar(n, umbral=0.0):
    return 1 if n >= umbral else 0

def evaluar_dataset(datos, umbral=0.0):
    res = datos.copy()
    res['n'] = res.apply(lambda r: calcular_n(r['horas_uso'], r['temperatura'], r['vibracion']), axis=1)
    res['pred'] = res['n'].apply(lambda val: clasificar(val, umbral))

    def clasif(r):
        if r['resultado_real'] == 1 and r['pred'] == 1: return 'VP'
        if r['resultado_real'] == 1 and r['pred'] == 0: return 'FN'
        if r['resultado_real'] == 0 and r['pred'] == 1: return 'FP'
        return 'VN'

    res['tipo'] = res.apply(clasif, axis=1)

    vp = len(res[res.tipo == 'VP'])
    fn = len(res[res.tipo == 'FN'])
    fp = len(res[res.tipo == 'FP'])
    vn = len(res[res.tipo == 'VN'])
    total = len(res)

    return {
        'df': res[['caso', 'horas_uso', 'temperatura', 'vibracion', 'n', 'pred', 'resultado_real', 'tipo']],
        'matriz': {'VP': vp, 'FN': fn, 'FP': fp, 'VN': vn, 'Total': total},
        'metricas': {
            'Exactitud': (vp + vn) / total if total > 0 else 0,
            'Precisión': vp / (vp + fp) if (vp + fp) > 0 else 0,
            'Sensibilidad': vp / (vp + fn) if (vp + fn) > 0 else 0,
            'Especificidad': vn / (vn + fp) if (vn + fp) > 0 else 0
        }
    }


### 3. Evaluación en Entrenamiento
Comprobamos que las cuentas cierren con los datos observados durante el diseño.


In [ ]:
res_ent = evaluar_dataset(entrenamiento)
display(res_ent['df'])

print("MATRIZ DE ENTRENAMIENTO")
print(res_ent['matriz'])
print("MÉTRICAS")
for k, v in res_ent['metricas'].items():
    print(f"{k:<15}: {v*100:.1f}%")


,caso,horas_uso,temperatura,vibracion,n,pred,resultado_real,tipo
0,1,4.1,58.0,2.3,-65.8,0,0,VN
1,2,9.5,66.0,3.9,-31.0,0,0,VN
2,3,11.8,84.0,6.8,20.6,1,1,VP
3,4,3.2,52.0,1.6,-80.6,0,0,VN
4,5,7.4,71.0,5.2,-17.2,0,0,VN
5,6,12.6,61.0,2.9,-39.8,0,0,VN
6,7,8.9,88.0,7.4,24.8,1,1,VP
7,8,5.6,63.0,3.1,-49.8,0,0,VN
8,9,10.2,74.0,4.4,-16.6,0,0,VN
9,10,6.3,91.0,8.1,29.6,1,1,VP


MATRIZ DE ENTRENAMIENTO
{'VP': 4, 'FN': 0, 'FP': 0, 'VN': 10, 'Total': 14}
MÉTRICAS
Exactitud      : 100.0%
Precisión      : 100.0%
Sensibilidad   : 100.0%
Especificidad  : 100.0%


### 4. Evaluación en Prueba
Evaluación a ciegas para medir la capacidad de generalización sobre casos no vistos.


In [ ]:
res_pru = evaluar_dataset(prueba)
display(res_pru['df'])

print("MATRIZ DE PRUEBA")
print(res_pru['matriz'])
print("MÉTRICAS")
for k, v in res_pru['metricas'].items():
    print(f"{k:<15}: {v*100:.1f}%")


,caso,horas_uso,temperatura,vibracion,n,pred,resultado_real,tipo
14,15,9.1,70.0,4.6,-20.8,0,0,VN
15,16,10.7,86.0,7.0,22.4,1,1,VP
16,17,3.9,60.0,2.5,-62.2,0,0,VN
17,18,7.7,81.0,6.4,5.4,1,0,FP
18,19,12.1,76.0,6.3,8.2,1,1,VP
19,20,5.1,57.0,2.2,-65.8,0,0,VN


MATRIZ DE PRUEBA
{'VP': 2, 'FN': 0, 'FP': 1, 'VN': 3, 'Total': 6}
MÉTRICAS
Exactitud      : 83.3%
Precisión      : 66.7%
Sensibilidad   : 100.0%
Especificidad  : 75.0%


### 5. Comparación con Línea Base Trivial
Contraste contra un clasificador ingenuo que predice siempre la clase mayoritaria (NO parar).


In [ ]:
base_ent = len(entrenamiento[entrenamiento.resultado_real == 0]) / len(entrenamiento)
base_pru = len(prueba[prueba.resultado_real == 0]) / len(prueba)

print(f"Línea Base Trivial ('Siempre NO'):")
print(f"  • Exactitud Entrenamiento : {base_ent*100:.1f}% | Sensibilidad: 0.0%")
print(f"  • Exactitud Prueba        : {base_pru*100:.1f}% | Sensibilidad: 0.0%")
print()
print(f"Neurona Artificial:")
print(f"  • Exactitud Entrenamiento : {res_ent['metricas']['Exactitud']*100:.1f}% (+{(res_ent['metricas']['Exactitud']-base_ent)*100:.1f} pts) | Sensibilidad: 100.0%")
print(f"  • Exactitud Prueba        : {res_pru['metricas']['Exactitud']*100:.1f}% (+{(res_pru['metricas']['Exactitud']-base_pru)*100:.1f} pts) | Sensibilidad: 100.0%")


Línea Base Trivial ('Siempre NO'):
  • Exactitud Entrenamiento : 71.4% | Sensibilidad: 0.0%
  • Exactitud Prueba        : 66.7% | Sensibilidad: 0.0%

Neurona Artificial:
  • Exactitud Entrenamiento : 100.0% (+28.6 pts) | Sensibilidad: 100.0%
  • Exactitud Prueba        : 83.3% (+16.7 pts) | Sensibilidad: 100.0%


### 6. Ejecución en Vivo para la Defensa
Función interactiva para evaluar en tiempo real cualquier caso que proponga el docente.


In [ ]:
def resolver_caso_en_vivo_interactivo(umbral=0.0):
    try:
        print("Modo Prueba en Vivo")
        h_str = input("Horas de uso desde service (cientos de h, ej: 8.5): ")
        t_str = input("Temperatura del rodamiento (°C, ej: 85): ")
        v_str = input("Vibración mecánica (mm/s, ej: 6.5): ")

        # Convierte coma a punto por si se ingresa con formato latino
        horas = float(h_str.replace(',', '.'))
        temp = float(t_str.replace(',', '.'))
        vib = float(v_str.replace(',', '.'))

        n = calcular_n(horas, temp, vib)
        decision = clasificar(n, umbral)

        print(f"Resultado del Caso")
        print(f"Entradas: Horas = {horas} ({horas*100:.0f} h) | Temp = {temp}°C | Vib = {vib} mm/s")
        print(f"Cuenta  : 2.0*({horas}) + 1.0*({temp}) + 10.0*({vib}) - 155.0 = {n:.2f}")
        print(f"Decisión: {'SÍ (PARAR PARA SERVICE)' if decision == 1 else 'NO (CONTINUAR)'}\n")

    except ValueError:
        print("\nError: Asegurate de ingresar números válidos.")

# Para ejecutar en la defensa:
resolver_caso_en_vivo_interactivo()



 Modo Prueba en Vivo

 Resultado del Caso
Entradas: Horas = 8.5 (850 h) | Temp = 85.0°C | Vib = 6.5 mm/s
Cuenta  : 2.0*(8.5) + 1.0*(85.0) + 10.0*(6.5) - 155.0 = 12.00
Decisión: SÍ (PARAR PARA SERVICE)

